# P1 — What is over-squashing?

A message-passing GNN updates each node by mixing in its neighbours, one hop at a time. When many paths
funnel into a node, all their messages are crushed into one fixed-width vector and signal is lost. We make
this precise on a **bottleneck** graph you can build with `aiq-quivers`.

## 1. Message passing moves one hop per layer

To reach a node `g` hops away you need `g` layers — and everything along the way is re-squashed at each step.

<img src="../figs-theory/en/E1a_message_one_hop.svg" width="760"/>

## 2. The bottleneck and the formula K·M^d

<img src="../figs-theory/en/T2_bottleneck_formula.svg" width="760"/>

In [ ]:
# Build the bottleneck with aiq-quivers' helper and look at it.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash import viz
K, M = 4, 3
data = make_bottleneck_graph(K, M, depth=2, generator=torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='K sources -> d layers of M -> target')

## 3. A fixed-width vector overflows

The walk count into the target grows like `K·M^d`, but the target's vector stays the same size.

<img src="../figs-theory/en/E1b_vector_overflow.svg" width="760"/>

In [ ]:
# Count the messages forced into the target, for growing depth (via walk operators).
from oversquash.ideal_bridge import walk_operators
for d in [1, 2, 3, 4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'depth {d}: {int(raw[d][:, t].sum()):4d} messages squashed into one vector  (~ K*M^d)')

## 4. The walk count explodes with distance

<img src="../figs-theory/en/E1c_explosion.svg" width="760"/>

**Next (P2):** to reason about all these paths exactly, we need the path algebra `kQ`.